# Agentic VQA Pipeline — Rerun Failed Questions (Phi-3.5-Vision)

Notebook dedicato al **recupero mirato delle domande andate in errore** nell'esperimento Phi-3.5-Vision.

### ⚡ Come funziona
1. Carica il file dei risultati parziale/con errori (da `/kaggle/input/...` o `/kaggle/working/...`).
2. Rileva automaticamente solo le domande fallite (`CUDA OOM`, `Error`, `EXTRACTION_FAILURE`).
3. Rielabora **esclusivamente quelle domande** usando Phi-3.5-Vision quantizzato a **8-bit** per evitare OOM.
4. Salva il file JSON completo con tutte le domande riparate.

### 🛠️ Differenze dal run originale
- **Quantizzazione INT8**: il modello occupa ~7.5GB invece di ~15GB, lasciando ~8.5GB per KV-cache e immagini.
- **Compatibilità patches**: include i fix per `DynamicCache` e il workaround immagine fittizia per prompt text-only.

In **Notebook options** abilita:
- Accelerator: **GPU T4 x2**
- Internet: **On**

In [ ]:
# ---- PARAMETRI DELL'ESPERIMENTO ----
PROJECT_SOURCE = ""  # Lascia vuoto per clonare/aggiornare l'ultimo commit da Git
GIT_REPOSITORY_URL = "https://github.com/matteo-petrelli/Agentic-VQA-Pipeline"
GIT_REF = "main"

# File di input con i risultati da riparare (Kaggle Input o Working)
INPUT_RESULT_JSON = "/kaggle/input/datasets/matteopetrelli/results-phi35/unanswerability_diagnostic_results_phi35.json"

# File di output dove salvare i risultati completi e riparati
OUTPUT_RESULT_JSON = "/kaggle/working/unanswerability_diagnostic_results_phi35_recovered.json"

IMAGE_DIR = "/kaggle/input/datasets/matteopetrelli/dude-train/content/DUDE_train-val-test_binaries/images/train"

# Modello e Profilo
HF_MODEL_NAME = "microsoft/Phi-3.5-vision-instruct"
PROMPT_PROFILE = "phi35_focused"

# Assegnazione GPU
EVIDENCE_GPU = 0
VLM_GPU = 1
ALLOW_SINGLE_GPU_FALLBACK = True

## 1. Setup Ambiente e Aggiornamento Codice da Git

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
_clone_dir = WORKING_DIR / "Agentic-VQA-Pipeline"

# Clona o aggiorna all'ultimo commit da GitHub per avere tutti i fix attivi
if GIT_REPOSITORY_URL:
    if (_clone_dir / ".git").is_dir():
        print(f"Pulling latest changes in {_clone_dir}...")
        subprocess.run(["git", "-C", str(_clone_dir), "pull", "origin", GIT_REF], check=False)
    elif not (_clone_dir / "agentic_pipeline.py").is_file():
        print(f"Cloning {GIT_REPOSITORY_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_REPOSITORY_URL, str(_clone_dir)], check=True)
    if str(_clone_dir) not in sys.path:
        sys.path.insert(0, str(_clone_dir))

from kaggle_utils import find_project, setup_environment

PROJECT_DIR = find_project(PROJECT_SOURCE, GIT_REPOSITORY_URL, GIT_REF, WORKING_DIR)
setup_environment(PROJECT_DIR)

## 2. Rilevamento GPU

In [ ]:
from kaggle_utils import detect_gpus

EVIDENCE_DEVICE, VLM_GPU_DEVICE = detect_gpus(EVIDENCE_GPU, VLM_GPU, ALLOW_SINGLE_GPU_FALLBACK)
HF_VLM_DEVICE = f"cuda:{VLM_GPU_DEVICE}" if isinstance(VLM_GPU_DEVICE, int) else VLM_GPU_DEVICE
print(f"Evidence device: {EVIDENCE_DEVICE}")
print(f"VLM device: {HF_VLM_DEVICE}")

## 3. Inizializzazione Configurazione e DocumentEngine

Configura il backend **transformers** con quantizzazione **INT8** per ridurre l'uso di VRAM
da ~15GB a ~7.5GB ed evitare gli errori CUDA OOM.

In [ ]:
import json, shutil
import config
from diagnostic_agent.engine import DocumentEngine
from diagnostic_agent.profiles import resolve_prompt_profile

# Copia il file di input nel file di output di destinazione se diverso
in_path = Path(INPUT_RESULT_JSON)
out_path = Path(OUTPUT_RESULT_JSON)

if not in_path.exists():
    matches = list(Path("/kaggle/input").glob(f"**/{in_path.name}"))
    if matches:
        in_path = matches[0]
        print(f"File trovato in Kaggle Input: {in_path}")
    else:
        raise FileNotFoundError(f"File non trovato: {INPUT_RESULT_JSON}")

if in_path.resolve() != out_path.resolve():
    out_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(in_path, out_path)
    print(f"Copiato file sorgente in: {out_path}")

# Configurazione Pipeline - Backend Transformers con quantizzazione 8-bit
config.IMAGE_DIR = IMAGE_DIR
config.OUTPUT_JSON_PATH = str(out_path)
config.EVIDENCE_DEVICE = EVIDENCE_DEVICE
config.VLM_NUM_CTX = 32768
config.VLM_MAX_TOKENS = 1536

# Backend Transformers (non Ollama)
config.VLM_BACKEND = "transformers"
config.HF_VLM_MODEL_NAME = HF_MODEL_NAME
config.HF_VLM_DEVICE = HF_VLM_DEVICE
config.HF_VLM_DTYPE = "float16"
config.HF_VLM_QUANTIZE = "4bit"  # "4bit" o "8bit". Usa 4bit per evitare OOM estremi
config.MAX_IMAGE_SIZE = 1024
config.HF_VLM_CACHE_DIR = "/tmp/hf_cache"

# Prompt profile
config.PROMPT_PROFILE = PROMPT_PROFILE
config.OLLAMA_VLM = "phi3.5-vision"  # Per logging/profile resolution

engine = DocumentEngine()
active_profile = resolve_prompt_profile(config.OLLAMA_VLM, PROMPT_PROFILE)
print(f"\nDocumentEngine caricato con successo.")
print(f"Profilo attivo: {active_profile.name}")
print(f"Quantizzazione VLM: {config.HF_VLM_QUANTIZE}")

## 4. Caricamento del modello Phi-3.5-Vision (INT8)

Carica esplicitamente il modello quantizzato a 8-bit sulla GPU dedicata.

In [ ]:
engine.setup_transformers_vlm()
print(f"\nPhi-3.5-Vision loaded successfully on {config.HF_VLM_DEVICE} (INT8).")

import subprocess
subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free", "--format=csv"],
    check=True,
)

## 5. Esecuzione del Rerun Mirato (Solo Domande Fallite)

L'agente ispeziona il file, identifica solo le domande che hanno riscontrato errori
(CUDA OOM, Error 400, EXTRACTION_FAILURE con trace vuoto) ed esegue il re-testing.

In [ ]:
from scripts.run_experiments import main as run_pipeline

# Esecuzione mirata in modalità retry_failed=True
run_pipeline(
    model_name=config.OLLAMA_VLM,
    profile_name=active_profile.name,
    engine=engine,
    retry_failed=True,
)

## 6. Verifica dei Risultati e Statistiche Post-Recupero

In [ ]:
import json
from collections import Counter

with open(OUTPUT_RESULT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

items = data.get("corrupted_questions", [])
total = len(items)

ans_dist = Counter()
causes_dist = Counter()
err_count = 0
expl_count = 0
expl_lengths = []

for it in items:
    res = it.get("agentic_result", {})
    ans = res.get("answerability", "unknown")
    cause = res.get("primary_cause") or "None"
    expl = res.get("cause_explanation") or ""
    final_ans = str(res.get("final_answer", ""))
    
    ans_dist[ans] += 1
    causes_dist[cause] += 1
    
    if "400" in final_ans or "Error" in final_ans:
        err_count += 1
    if expl.strip():
        expl_count += 1
        expl_lengths.append(len(expl.strip()))

print("="*60)
print("📊 REPORT FINALE POST-RECUPERO")
print("="*60)
print(f"Totale Domande: {total}")
print(f"Errori Residui: {err_count} ({err_count/total*100:.1f}%)")
print(f"Diagnosi unanswerable riuscite: {ans_dist['unanswerable']} ({ans_dist['unanswerable']/total*100:.1f}%)")
print(f"Spiegazioni generate: {expl_count} ({expl_count/total*100:.1f}%)")
if expl_lengths:
    print(f"Lunghezza media spiegazioni: {sum(expl_lengths)/len(expl_lengths):.1f} caratteri")

print("\nDistribuzione Cause Diagnosticate:")
for c, count in sorted(causes_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {c:30s}: {count:3d} ({count/total*100:.1f}%)")

print(f"\n✅ File completo e recuperato salvato in:\n   {OUTPUT_RESULT_JSON}")

### Pulizia memoria

Libera la memoria GPU al termine dell'esperimento.

In [ ]:
import gc
import torch

if hasattr(engine, '_hf_vlm_model') and engine._hf_vlm_model is not None:
    del engine._hf_vlm_model
    engine._hf_vlm_model = None
if hasattr(engine, '_hf_vlm_processor') and engine._hf_vlm_processor is not None:
    del engine._hf_vlm_processor
    engine._hf_vlm_processor = None
torch.cuda.empty_cache()
gc.collect()
print("Memoria GPU liberata.")